In [1]:
from Bio import Entrez, SeqIO
import os
import json

In [2]:
# Input variables
type = 'H7N9'
segment = 'HA'
accession = "NC_026425.1"

In [3]:
def download_gene_sequence(accession):
    """Download gene sequence in FASTA format using accession number"""
    # Fetch the sequence
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
    sequence = handle.read()
    handle.close()
    
    # Save to file
    output_fasta = os.path.join(output_dir, f"{accession}.fasta")
    with open(output_fasta, "w") as f:
        f.write(sequence)
    
    print(f"Sequence saved to {output_fasta}")
    return output_fasta

def download_gene_gff(accession):
    """Download GFF file for gene using accession number"""
    # First get the GI number or other identifiers if needed
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
    record = SeqIO.read(handle, "genbank")
    handle.close()
    
    # Fetch the GFF
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gff3", retmode="text")
    gff_content = handle.read()
    handle.close()
    
    # Save to file
    output_gff = os.path.join(output_dir, f"{accession}.gff")
    with open(output_gff, "w") as f:
        f.write(gff_content)
    
    print(f"GFF saved to {output_gff}")
    return output_gff

# Download and save a FASTA file and GFF file for a specific sequence accession number
output_dir = f'../results/{type}-{segment}/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
Entrez.email = "test@gmail.com"
fasta_file = download_gene_sequence(accession)
gff_file = download_gene_gff(accession)

Sequence saved to ../results/H7N9-HA/NC_026425.1.fasta
GFF saved to ../results/H7N9-HA/NC_026425.1.gff


In [4]:
# Fill out data for pathogen JSON
pathogen_json = {
  "schemaVersion": "3.0.0",
  "files": {
    "genomeAnnotation": os.path.basename(gff_file),
    "pathogenJson": "pathogen.json",
    "reference": os.path.basename(fasta_file),
  },
  "alignmentParams": {
    "minSeedCover": 0.1
  }
}

# Write pathogen_json to a JSON file
json_file_path = os.path.join(output_dir, "pathogen.json")
with open(json_file_path, 'w') as f:
    json.dump(pathogen_json, f, indent=2)  # indent=2 makes the JSON file nicely formatted

print(f"Pathogen JSON saved to {json_file_path}")

Pathogen JSON saved to ../results/H7N9-HA/pathogen.json
